# LLM Observability Hackathon

You are the investigator. Each activity produces traces in your local Langfuse
project; open Langfuse at `http://localhost:3000` after the compose stack is up.

## Setup

Required environment variables:

- `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_BASE_URL`
- `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `OPENAI_MODEL`

The notebook fails loudly when any variable is missing. Model calls use
`temperature=0` and fixed inputs; trace ids and the run id vary per run.

In [1]:
import os, json, re, time, uuid
from langfuse import Langfuse, propagate_attributes
from langfuse.openai import OpenAI

REQUIRED = ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL",
            "OPENAI_BASE_URL", "OPENAI_API_KEY", "OPENAI_MODEL"]
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise RuntimeError("Missing environment variables: " + ", ".join(missing))

In [2]:
langfuse = Langfuse()
try:
    authenticated = langfuse.auth_check()
except Exception as exc:
    raise RuntimeError(f"Langfuse authentication failed for {os.environ['LANGFUSE_BASE_URL']}: {exc}") from exc
if not authenticated:
    raise RuntimeError(f"Langfuse authentication failed for {os.environ['LANGFUSE_BASE_URL']}")
client = OpenAI(base_url=os.environ["OPENAI_BASE_URL"], api_key=os.environ["OPENAI_API_KEY"])
MODEL = os.environ["OPENAI_MODEL"]
RUN_ID = uuid.uuid4().hex[:8]

def show_trace(trace_id):
    url = langfuse.get_trace_url(trace_id=trace_id)
    print(f"trace: {url or trace_id}")
    return url or trace_id

print("run id:", RUN_ID)

run id: f187667a


## 1. Activity 1 — RAG with and without filter

**Goal:** Compare a RAG answer produced with unfiltered retrieval against one produced with a tier filter.

**Expected observation:** The unfiltered run retrieves a policy for another tier and answers from it; the filtered run retrieves the Basic policy.

**Deliverable:** Both trace URLs, the retrieved document ids, and the answer difference.

## 2. Activity 2 — Output format

**Goal:** Compare an unconstrained prose answer with a constrained JSON answer for a downstream booking parser.

**Expected observation:** The prose answer fails validation; the constrained answer parses.

**Deliverable:** Validation results and trace URLs.

## 3. Activity 3 — Fix hallucination

**Goal:** Find the cause of a wrong baggage answer in the trace and fix the system prompt.

**Expected observation:** The broken run answers two checked bags while the retrieved Gold policy says one.

**Deliverable:** Root cause and the fixed answer.

## 4. Activity 4 — Debug tool calling

**Goal:** Diagnose a failing tool call from trace data and repair the tool.

**Expected observation:** The tool span shows the arguments and a KeyError; the fixed run returns the booking.

**Deliverable:** Failing span details and the repaired answer.

## 5. Wrap-up

Flush the Langfuse client so the last traces are delivered. Submit the trace URLs
and your root causes for Activities 3 and 4.

In [3]:
langfuse.flush()
print("flushed; run id:", RUN_ID)

flushed; run id: f187667a
